# Collect AVH Data for Analysis
The following code brings together data generated and collected by Feng, Changye, and George into a single dataframe for future analysis. Right now the focus is on transcript level information for the sandwich analysis.

TODO

    1) How to handle NAs? Can implement smarter NA handling by only removing rows that have NAs in the columns we care about
    2) Log of WER
    3) Add in pause proportion
    4) Do not use SNR for WER

## Import libraries and functions

In [168]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
# import statsmodels.api as sm
# from statsmodels.regression.mixed_linear_model import MixedLM
import numpy as np
from io import StringIO
from pathlib import Path

# directory_path = Path("another_new_directory")
  
# directory_path.mkdir(parents=True, exist_ok=True)
# print(f"Directory '{directory_path}' ensured to exist.")
out_folder = "/edata/obdw/sandwich_analysis_data/"

In [169]:
# Import shared utilities
import sys
import importlib
sys.path.insert(0, '/home/NETID/emd5')
import avh_utils
# Reload in case the module has been updated
importlib.reload(avh_utils)
from avh_utils import decode_variable_name, data_dictionary

In [170]:
def process_uid(series: pd.SparseDtype) -> pd.Series:
    """
    Extract unique pid from file names in a pandas Series. From Changye Li

    Args:
        series (pd.Series): Series containing file names of the transcripts

    Returns:
        pd.Series: Series with processed PIDs as integers
    """
    # Split the strings and get the first element (pid part)
    pids = series.str.split('-').str[0]
    
    # Clean up the pid strings
    pids = (pids.str.rstrip('@avh')
                .str.lstrip('u')
                .str.lstrip('0')
                .astype(int))
    
    return pids

## Read in Data

In [171]:
ptcpt_baseline_path = "/edata/changye/restore/avh-data/avh_record_baseline_kwargs.jsonl"# I wil use this file as the main
#this was old path
#ptcpt_baseline_path = "/edata/changye/avh-data/avh_record_baseline_kwargs.jsonl"# I wil use this file as the main

# rcd_whisperx_scores_path = "/edata/feng/avh/whisperx_baseline_with_time_wer_english_full.json"
# old path
rcd_whisperx_scores_path = "/edata/feng/avh/whisperx_merged_with_manual.json" #"/edata/feng/avh/whisperx_full_cleaned.json"
rcd_snr_path = "/edata/changye/restore/avh-new/data/snr.csv"
#rcd_snr_path = "/edata/changye/avh-new/data/snr.csv"
rcd_mos_path = "/edata/changye/restore/avh-data/pred_mos.csv"
rcd_coh_recording_level_path = "/edata/george/debias_merged_new.csv"




In [172]:
baseline_df = pd.read_json(ptcpt_baseline_path, lines=True)
baseline_df = baseline_df.set_index("pid", drop=True).rename_axis("file")
baseline_df = baseline_df.rename(columns={"uid":"pid"})
baseline_df.shape
#list(baseline_df.columns)

(2904, 128)

In [173]:
baseline_df = pd.read_json(ptcpt_baseline_path, lines=True)
baseline_df = baseline_df.set_index("pid", drop=True).rename_axis("file")
baseline_df = baseline_df.rename(columns={"uid":"pid"})

rcd_whisperx_scores_df = pd.read_json(rcd_whisperx_scores_path, lines=True)
rcd_whisperx_scores_df = rcd_whisperx_scores_df.set_index("file", drop=True)#.rename_axis("file")
# rcd_whisperx_scores_df["pid"] = process_uid(rcd_whisperx_scores_df.pid)


rcd_snr_df = pd.read_csv(rcd_snr_path)
rcd_snr_df = rcd_snr_df.set_index("file")
rcd_snr_df["pid"] = process_uid(rcd_snr_df.index)

rcd_mos_df = pd.read_csv(rcd_mos_path)
rcd_mos_df = pd.DataFrame(rcd_mos_df.groupby(['file_name'])['pred_mos'].mean())
rcd_mos_df["pid"] = process_uid(rcd_mos_df.index)


rcd_level_coh_df = pd.read_csv(rcd_coh_recording_level_path)# Similar to above but for recordings.
rcd_level_coh_df["file"] = rcd_level_coh_df.file.apply(lambda x: x.strip(".txt"))
rcd_level_coh_df = rcd_level_coh_df.set_index("file")
rcd_level_coh_df["pid"] = process_uid(rcd_level_coh_df.index)


In [174]:
list(rcd_whisperx_scores_df.columns)

['segments',
 'prediction',
 'time_diffs',
 'pause_proportion',
 'pid',
 'record',
 'text',
 'audio',
 'wer',
 'cer']

In [175]:
#list(rcd_level_coh_df.columns)

In [176]:
list(rcd_snr_df.columns)

['snr', 'pid']

In [177]:
print("Baseline scores shape", baseline_df.shape)
print("Whisper scores shape", rcd_whisperx_scores_df.shape)
print("SNR shape", rcd_snr_df.shape)
print("MOS shape", rcd_mos_df.shape)
print("COH shape", rcd_level_coh_df.shape)

Baseline scores shape (2904, 128)
Whisper scores shape (2994, 10)
SNR shape (3003, 2)
MOS shape (2981, 2)
COH shape (3003, 133)


## Clean Data

### Check transcript file names

In [178]:
baseline_files = set(baseline_df.index)
whisper_files = set(rcd_whisperx_scores_df.index)
snr_files = set(rcd_snr_df.index)
mos_files = set(rcd_mos_df.index)
coh_files = set(rcd_level_coh_df.index)

file_names_union = whisper_files.union(snr_files).union(mos_files).union(coh_files).union(baseline_files)
file_names_intersection = whisper_files.intersection(snr_files).intersection(mos_files).intersection(coh_files).intersection(baseline_files)
print("There are a total of {:,} file names and only {:,} at the intersection".format(len(file_names_union), len(file_names_intersection)))

There are a total of 3,007 file names and only 2,886 at the intersection


### Remove duplicated columns
Not all will be removed but this should clean it up some

In [179]:
def remove_duplicated_columns(main_df, secondary_df, extra_cols_to_drop, protected_cols = set(["pid"])):
    """
    Description: removes duiplicated columns from secondary_df which main_df already contains
    Input:
        main_df (pandas df): dataframe
        secondary_df (pandas df): dataframe
        extra_cols_to_drop (list or set): Additional column names to drop from extra cols. Useful if you know cols
            are duplicates but with different names
        protected_cols (list or set): Columns not to drop.  THIS IS NO LONGER USED AND DOES NOT WORK ANYWAY
    Output:
        updated_df: (pandas df): same as secondary_df, but with duplicated columns removed
    TODO
        1)
    """

    intersection_columns = set(main_df).intersection(secondary_df).union(extra_cols_to_drop)
    print(intersection_columns)
    print("cols dropped", len(intersection_columns))
    updated_df = secondary_df.drop(intersection_columns, axis=1)
    print("new shape", updated_df.shape)
    return(updated_df)

In [180]:
coh_cols_to_drop = ['contact',
 'temper-outbursts',
 'feeling-blocked',
 'worrying-too-much',
 'easily-hurt',
 'feeling-watched',
 'feeling-tense',
 'heavy-feelings-in-arms-legs',
 'feeling-nervous',
 'feeling-lonely',
 'frequency',
 'bad-voices',
 'volume-of-voices',
 'voices-length',
 'interference-in-activities',
 'distressing-voices',
 'worthless-useless-voices',
 'clarity-of-voices',
 'follow-voices-orders',
 'time-of-day-of-voices',
 'social-situations',
 'where-are-the-voices',
 'typical-week',
 'if-no-explanation',
 'work-school-disruption',
 'social-leisure-disruption',
 'home-family-disruption',
 'school-work-missed',
 'less-productive-days',
#  'not-worked-for-other-reasons',# not removed since this is one George suggested to keep
 'little-interest-or-pleasure',
 'feeling-depressed',
 'trouble-sleeping',
 'feeling-tired',
 'appetite',
 'feeling-bad-about-self',
 'trouble-concentrating',
 'slow-fast-speaking',
 #'suicidal-thoughts',
 'impact-on-your-life', 'text']

In [181]:
print("Removing columns from coh df")
rcd_level_coh_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_level_coh_df, extra_cols_to_drop=coh_cols_to_drop)
print("\n\nRemoving columns from whisperx df")
rcd_whisperx_scores_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_whisperx_scores_df, extra_cols_to_drop=["text"])
print("\n\nRemoving columns from mos df")
rcd_mos_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_mos_df, extra_cols_to_drop=[])
print("\n\nRemoving columns from snr df")
rcd_snr_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_snr_df, extra_cols_to_drop=[])

Removing columns from coh df
{'feeling-blocked', 'impact-on-your-life', 'feeling-lonely', 'comp_q4', 'data_plan', 'how-often', 'email', 'ah_freq', 'type-of-treatments', 'time-of-day-of-voices', 'contact', 'smartphone', 'feeling-tired', 'location', 'comp_q5', 'phone', 'gender', 'diagnoses', 'text', 'clarity-of-voices', 'employment-status', 'comp_q2', 'temper-outbursts', 'follow-voices-orders', 'social-situations', 'interference-in-activities', 'feeling-bad-about-self', 'feeling-nervous', 'bad-voices', 'if-no-explanation', 'hispanic', 'feeling-depressed', 'distressing-voices', 'availability', 'education', 'in_person', 'typical-week', 'school-work-missed', 'age', 'little-interest-or-pleasure', 'days of data', 'who-knows', 'tablet', 'home-family-disruption', 'where-are-the-voices', 'appetite', 'worthless-useless-voices', 'living-status', 'used-treatment', 'easily-hurt', 'trouble-sleeping', 'computer', 'substances-used', 'heavy-feelings-in-arms-legs', 'comp_q1', 'feeling-watched', 'volume-o

## Merge Data

In [182]:
main_analysis_df = pd.concat([baseline_df, rcd_whisperx_scores_df, rcd_snr_df, rcd_mos_df, rcd_level_coh_df], axis=1, join="inner")
assert main_analysis_df.shape[0] == len(file_names_intersection)
print(main_analysis_df.shape)
#print(main_analysis_df.columns)
#print(main_analysis_df.columns.tolist())

(2886, 190)


In [183]:
# Replace clinical sentinel values (999) with NA to match sandwich_script.r preprocessing.
sentinel_value = 999
sentinel_cols = [
    "scl-avg-global-score",
    "phq9-total",
    "hpsvq-total-score",
    # Alternate dotted names for compatibility with exported CSV conventions
    "scl.avg.global.score",
    "phq9.total",
    "hpsvq.total.score",
]

existing_sentinel_cols = [c for c in sentinel_cols if c in main_analysis_df.columns]

if not existing_sentinel_cols:
    print("No sentinel target columns found in main_analysis_df.")
else:
    sentinel_summary = []
    for col in existing_sentinel_cols:
        if pd.api.types.is_numeric_dtype(main_analysis_df[col]):
            n_sentinel = int((main_analysis_df[col] == sentinel_value).sum())
            if n_sentinel > 0:
                main_analysis_df.loc[main_analysis_df[col] == sentinel_value, col] = np.nan
            sentinel_summary.append((col, n_sentinel, int(main_analysis_df[col].isna().sum())))

    print("Sentinel replacement summary (column, replaced_999_count, final_na_count):")
    for row in sentinel_summary:
        print(row)

Sentinel replacement summary (column, replaced_999_count, final_na_count):
('scl-avg-global-score', 4, 5)
('phq9-total', 27, 28)
('hpsvq-total-score', 0, 1)


In [184]:
# Check that each participant has consistent location data across all transcripts
# Assumes you have a DataFrame with 'pid' as participant ID and location columns (e.g., 'PrimaryRUCA', 'RPL_THEMES', etc.)

#ToD0: location isn't merged in this high, move cell after location merge
location_columns = [
    'PrimaryRUCA', 'RPL_THEMES', 'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4',
    'doctor_count', 'pharmacy_count', 'hospital_count', 'park_count', 'bus_station_count', 'supermarket_count', 'cluster'
]

# Replace 'main_analysis_df' with the actual DataFrame variable name containing these columns
df_location = main_analysis_df[['pid'] + [col for col in location_columns if col in main_analysis_df.columns]].copy()

# Group by participant and check for unique location rows
inconsistent_pids = []
for pid, group in df_location.groupby('pid'):
    # Drop duplicate location rows for this participant
    unique_locs = group[[col for col in location_columns if col in group.columns]].drop_duplicates()
    if len(unique_locs) > 1:
        inconsistent_pids.append(pid)
    if pid == 8: 
        print(unique_locs)


if inconsistent_pids:
    print(f"WARNING: Found {len(inconsistent_pids)} participants with inconsistent location data across transcripts.")
    print("Participant IDs with inconsistent location data:", inconsistent_pids)
else:
    print("All participants have consistent location data across transcripts.")

#ToDo: moved this code around, doesn't work
# Have to redo the above code to check for consistency of location data across transcripts. I moved it around and it doesn't work anymore, not sure why.


Empty DataFrame
Columns: []
Index: [u00000008@avh-20180311-1, u00000008@avh-20180329-1]
Participant IDs with inconsistent location data: [3, 8, 50, 55, 57, 79, 88, 90, 105, 115, 118, 123, 134, 162, 170, 235, 240, 253, 275, 281, 305, 313, 333, 341, 366, 373, 374, 382, 383, 392, 396, 409, 413, 441, 443, 448, 449, 450, 457, 472, 478, 485, 500, 509, 518, 520, 564, 581, 600, 606, 649, 652, 669, 677, 681, 688, 703, 705, 706, 739, 747, 768, 796, 802, 803, 814, 815, 827, 831, 844, 847, 848, 866, 879, 896, 919, 928, 933, 949, 960, 962, 963, 967, 993, 1008, 1010, 1017, 1020, 1035, 1037, 1042, 1063, 1081, 1088, 1155, 1166, 1172, 1179, 1204, 1206, 1211, 1228, 1247, 1248, 1251, 1252, 1261, 1283, 1293, 1295, 1296, 1319, 1322, 1325, 1350, 1351, 1352, 1358, 1363, 1377, 1396, 1405, 1425, 1435, 1441, 1442, 1450, 1460, 1461, 1489, 1493, 1499, 1501, 1515, 1517, 1524, 1529, 1537, 1553, 1558, 1574, 1580, 1595, 1597, 1607, 1621, 1639, 1643, 1644, 1651, 1665, 1679, 1682, 1683, 1686, 1690, 1691, 1706, 1714, 17

In [185]:
# Explore a few participants with inconsistent location data across transcripts
# Pick a few participant IDs from inconsistent_pids
example_pids = inconsistent_pids[:3]  # You can change the number or pick specific IDs

# for pid in example_pids:
#     print(f"\nParticipant {pid} - Location data across transcripts:")
#     display(df_location[df_location['pid'] == pid])
#     # Show unique location rows for this participant
#     unique_locs = df_location[df_location['pid'] == pid][[col for col in location_columns if col in df_location.columns]].drop_duplicates()
#     #print(f"Unique location rows for participant {pid}:")
#     #display(unique_locs)


## Remove NA
I am ignoring certain columns I do not expect to use. Like many of the coherence columns. Otherwise we would be dropping many columns with NAs that we would never use.

In [186]:
columns2ignore_na = ["not-worked-for-other-reasons"]
for column_name in main_analysis_df.columns:
    if "Coherence" in column_name and column_name != "sentCoherenceSentBertCumulativeCentroid":# the one we care about is sentCoherenceSentBertCumulativeCentroid
        columns2ignore_na.append(column_name)
# Code below just to look at columns with na
#  for x, y in zip(main_analysis_df.columns, list(main_analysis_df.isna().sum())):
#     if x in columns2ignore_na:
#         continue
#     print(x, y)
main_analysis_df.drop(columns2ignore_na, axis=1, inplace=True)
main_analysis_df.dropna(axis=0, inplace=True)
# main_analysis_df


In [187]:
main_analysis_df.to_csv(out_folder + "main_merged_sandwich_analysis_data.csv")

In [188]:
temp = pd.read_csv(out_folder + "main_merged_sandwich_analysis_data.csv", index_col=0)

In [189]:
main_analysis_df.shape

(2832, 154)

## Identify Features for Analysis
The next step is to identify subsets of features to use for sandwich analysis. Listing the features and response variable out should not be hard for each analysis. Removing nan values will depend more on the response variable I believe, since nan categorical variables can be coded up in one-hot encoding. However it might be wise to remove rows with any nan values.

### Options for final cleaning

1) Remove nan values from each dataframe on a per analysis criteria
2) Remove nan values from the overall dataframe so each analysis is working with the same dataframe.
3) A hybrid approach of the above where rows are removed based off of nan values in the features from the overall dataframe and then nan values are removed in the response variable on a per analysis criteria.

## Feature Engineering
As of now, this is only one-hot encoding code I have from the previous analysis

In [190]:
#list(main_analysis_df.columns.tolist())

In [191]:
# Loop through all columns in X_basic_plus_clin_sdh_location_encoded except 'WER' and run a simple linear regression (WER ~ variable)

# Univariate regression results will be used to identify which variables are significantly associated with WER, 

# and to understand the direction and magnitude of these associations. 

# This will help us identify potential predictors of WER and inform our multivariate regression model later on.

import pandas as pd

import statsmodels.api as sm

from statsmodels.iolib.summary2 import summary_col



# Load the data if not already loaded

data_path = '/edata/obdw/sandwich_analysis_data/X_basic_plus_clin_sdh_location_encoded.csv'

df = pd.read_csv(data_path)



# Identify the WER column: try multiple candidates

wer_candidates = ['WER', 'wer', 'log_wer', 'Y_WER']

wer_col = next((col for col in wer_candidates if col in df.columns), None)

if not wer_col:

    raise ValueError(f'WER column not found in the data. Tried: {wer_candidates}. Available: {list(df.columns)}')



# Loop through all columns except WER

# ToDo: Indiviual Variables

# Bar Plots = green coding if significant, red if not, and show the beta values on the plot. This is important for understanding the direction and magnitude of the relationship, and for visually communicating the results to others.

# ToDo: add the sandwich correction to the regression to account for non-independence of observations within participants. This is important because we have multiple transcripts per participant, and we want to make sure our standard errors are correct.

regression_results = {}

for col in df.columns:

    if col == wer_col or col == 'pid':

        continue

    # Drop NA for the current variable and WER

    temp = df[[wer_col, col, 'pid']].dropna()

    if temp.empty:

        continue  # Skip if there's no data after dropping NA

    

    vals = temp[col]  # might be Series OR DataFrame

    if isinstance(vals, pd.DataFrame):

        vals = vals.iloc[:, 0]



    nuniq = vals.nunique(dropna=True)



    # If vals is a DataFrame, nuniq is a Series (one per column)

    if isinstance(nuniq, pd.Series):

        if (nuniq < 2).any():   # or .all(), depending on intent

            continue

    else:

        if nuniq < 2:

            continue

    # Need at least 2 clusters for clustered SEs.

    pid = temp.loc[:, 'pid']

In [192]:
import importlib
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from patsy.builtins import Q
import avh_utils
importlib.reload(avh_utils)

# ── Config ────────────────────────────────────────────────────────────────────
data_path   = '/edata/obdw/sandwich_analysis_data/X_basic_plus_clin_sdh_location_encoded.csv'
output_path = '/home/NETID/emd5/Univariate_Regression_from_X_basic_plus_clin_sdh_location_encoded.csv'

df = pd.read_csv(data_path)

# Identify WER column
wer_candidates = ['Y_WER', 'WER', 'wer', 'log_wer']
wer_col = next((c for c in wer_candidates if c in df.columns), None)
if not wer_col:
    raise ValueError(f'WER column not found. Available: {list(df.columns)}')

print(f"Running univariate regressions: WER column = '{wer_col}', "
      f"{df.shape[0]} rows, {len(df.columns) - 3} predictor candidates")

# ── Univariate regression loop (clustered SEs by pid) ─────────────────────────
rows    = []
skipped = []

for col in df.columns:
    if col in (wer_col, 'pid', 'Y_COH'):
        continue

    temp = df[[wer_col, col, 'pid']].dropna()

    if temp.empty or temp[col].nunique() < 2 or temp['pid'].nunique() < 2:
        skipped.append(col)
        continue

    try:
        safe_col = col.replace('"', '')
        formula  = f'{wer_col} ~ Q("{safe_col}")'
        model    = smf.ols(formula=formula, data=temp).fit(
            cov_type='cluster',
            cov_kwds={'groups': temp['pid'], 'use_correction': True}
        )
        # params.iloc[0] = intercept, params.iloc[1] = predictor coefficient
        rows.append({
            'Variable':      col,
            'Coefficient':   model.params.iloc[1],
            'p-value':       model.pvalues.iloc[1],
            'N':             int(model.nobs),
            'R-squared':     model.rsquared,
            'Adj R-squared': model.rsquared_adj,
            'F-statistic':   model.fvalue,
            'F p-value':     model.f_pvalue,
        })
    except Exception as e:
        skipped.append(f'{col} ({e})')

# ── Build results DataFrame ────────────────────────────────────────────────────
results_df = pd.DataFrame(rows)
results_df['Variable_Decoded'] = results_df['Variable'].apply(avh_utils.decode_variable_name)
results_df = results_df.sort_values('p-value').reset_index(drop=True)

# ── Save ───────────────────────────────────────────────────────────────────────
results_df.to_csv(output_path, index=False)
print(f"Saved {len(results_df)} rows → {output_path}")
if skipped:
    print(f"Skipped {len(skipped)}: {skipped[:5]}{'...' if len(skipped) > 5 else ''}")

# ── Summary ────────────────────────────────────────────────────────────────────
sig = results_df[results_df['p-value'] < 0.05]
print(f"\nSignificant (p < 0.05): {len(sig)} of {len(results_df)} variables")
print(sig[['Variable_Decoded', 'Coefficient', 'p-value', 'N']].to_string(index=False))


Running univariate regressions: WER column = 'Y_WER', 2832 rows, 83 predictor candidates
Saved 83 rows → /home/NETID/emd5/Univariate_Regression_from_X_basic_plus_clin_sdh_location_encoded.csv

Significant (p < 0.05): 32 of 83 variables
                               Variable_Decoded  Coefficient      p-value    N
                                  Gender: Other     0.908998 1.091281e-73 2832
                   Gender: Transgender (F to M)    -0.693507 9.347759e-44 2832
  Psychoactive Substance Use (Ketamine or Acid)    -0.813823 2.290925e-43 2832
                             Sexuality: Unknown     0.597541 7.248843e-33 2832
         Primaryruca: Small Town High Commuting    -0.562297 2.545155e-28 2832
                    Signal-to-Noise Ratio (SNR)    -0.042148 5.650154e-27 2832
                   Primaryruca: Small Town Core    -0.448064 3.431585e-19 2832
        Primaryruca: Metropolitan Low Commuting    -0.617298 1.906218e-11 2832
                 Primaryruca: Micropolitan Core    -0

In [193]:
def one_hot_encode(df, cols2encode, missing_label="__MISSING__", forced_drop=None, prefer_drop=("other", "unknown"), fallback="most_frequent", verbose=True):
    """One-hot encode columns while explicitly selecting a reference category to drop per column."""
    df_enc = df[cols2encode].fillna(missing_label).astype(str)
    forced_drop = forced_drop or {}

    def _resolve_requested_category(requested, categories):
        requested_norm = str(requested).strip().lower()
        for cat in categories:
            if cat.strip().lower() == requested_norm:
                return cat

        try:
            requested_num = float(requested)
        except (TypeError, ValueError):
            requested_num = None

        if requested_num is not None:
            for cat in categories:
                try:
                    if float(cat) == requested_num:
                        return cat
                except (TypeError, ValueError):
                    continue
            return None

        for cat in categories:
            if requested_norm in cat.strip().lower():
                return cat
        return None

    drop_values = []
    dropped_map = {}

    for col in cols2encode:
        value_counts = df_enc[col].value_counts()
        categories = list(value_counts.index.astype(str))
        chosen = None

        if col in forced_drop:
            chosen = _resolve_requested_category(forced_drop[col], categories)
            if chosen is None:
                print(f"[one_hot_encode] Requested drop '{forced_drop[col]}' for '{col}' was not found. Falling back to automatic rule.")

        if chosen is None:
            for token in prefer_drop:
                chosen = _resolve_requested_category(token, categories)
                if chosen is not None:
                    break

        if chosen is None:
            if fallback == "least_frequent":
                chosen = str(value_counts.idxmin())
            else:
                chosen = str(value_counts.idxmax())

        drop_values.append(chosen)
        dropped_map[col] = chosen

    if verbose:
        print(f"[one_hot_encode] Dropped reference categories: {dropped_map}")

    try:
        enc = OneHotEncoder(drop=drop_values, sparse=False, handle_unknown="ignore", dtype=int)
    except TypeError:
        enc = OneHotEncoder(drop=drop_values, sparse_output=False, handle_unknown="ignore", dtype=int)

    enc.fit(df_enc)
    one_hot_encoded_df = pd.DataFrame(
        enc.transform(df_enc),
        columns=enc.get_feature_names_out(),
        index=df.index
    )
    return one_hot_encoded_df


In [194]:
# Keep a stable reference to the upgraded encoder even if a legacy redefinition appears later.
one_hot_encode_v2 = one_hot_encode

In [195]:
# Bin education into levels
binmappers = {
                "education":{
                    2.0: 1.0,# 1.0 means up to codes for education - data dictionary for things like HS diploma, College degree, etc.
                    3.0: 1.0,
                    4.0: 1.0,
                    5.0: 2.0,
                    6.0: 2.0,
                    7.0: 3.0,
                    8.0: 3.0,
                    np.nan:np.nan,
                }
           }
def bin_age(age):
    """
    Description: Bins all age value from an pandas series. To be used in an apply function
    Inputs:
        ages (float): Float age
    Outputs:
        age_bin (int): Int category for age bin
    TODO:
        1)
    """
    if np.isnan(age):
        age_bin = 0.0
    elif age == 999.0:
        age_bin = 0.0
    # elif age < 18.0:
    #     age_bin = 0.0
    elif age < 30.0:
        age_bin = 1.0
    elif age < 45.0:
        age_bin = 2.0
    elif age < 65.0:
        age_bin = 3.0
    else:
        age_bin = 4.0
    return(age_bin)

def replaced_age_with_binning(df):
    """
    Description: Wrapper for bin_age where the original age columns is dropped
    Inputs:
        df (pandas df): Pandas dataframe
    Outputs:
        df (pandas df): Same as input, but with age removed, and binned_age added
    TODO:
        1)
    """
    temp_df = df.copy(deep=True)
    temp_df["binned_age"] = temp_df.age.apply(lambda x: bin_age(x))
    temp_df.drop("age", axis=1, inplace=True)
    return(temp_df)


def bin_substance_use(data_df):
    """
    Decription: Bins a participants substance use. Only done for heroin and opioids
    Inputs:
         data_df (pandas DF): Dataframe containing substance use columns below
            "opioids","marijuana","alcohol","steroids","cocaine","heroin","nicotine","meth","ketamine","acid","bath-salts"
    Outputs:
    TODO
        1)
    """

    data_copy_df = data_df.copy(deep=True)
    # All Types of Drug Use: Did not include "nicotine"
    all_types_drug_use = ["opioids","marijuana","alcohol","steroids","cocaine","heroin","meth","ketamine","acid","bath-salts"]
    substances_of_interest = ["opioids","heroin"]
    substance_marijuana = ["marijuana"]
    substance_alcohol = ["alcohol"]
    substance_stimulant = ["cocaine","meth","ketamine","acid","bath-salts"]
    substance_nicotine = ["nicotine"]
    substance_psychoactive = ["ketamine","acid"]
    # Drugs (opiods, cannabis, stimulants, psychoactive),alcohol, Nicotine
    substance_drugs = ["opioids","marijuana", "cocaine","heroin","meth","ketamine","acid","bath-salts"]
    # Backward-compatible alias for earlier typo/variable name usage
    subtance_psychoactive = substance_psychoactive
    # type_2 = ["cocaine","heroin","nicotine","meth","ketamine","acid","bath-salts"]

    # Binarize the relevant substance columns (1 -> 1.0, else 0.0)
    data_copy_df.loc[:, all_types_drug_use] = pd.DataFrame(data_copy_df.loc[:, all_types_drug_use] == 1, dtype=np.float64)
    if "nicotine" in data_copy_df.columns:
        data_copy_df.loc[:, "nicotine"] = pd.Series(data_copy_df.loc[:, "nicotine"] == 1, dtype=np.float64)

    def build_group_flag(df, cols):
        existing_cols = [c for c in cols if c in df.columns]
        if not existing_cols:
            return pd.Series(np.zeros(len(df), dtype=np.float64), index=df.index)
        return pd.Series(df.loc[:, existing_cols].sum(axis=1) > 0, dtype=np.float64)

    opiods_opiates_1_use = build_group_flag(data_copy_df, substances_of_interest)
    all_types_drug_use_flag = build_group_flag(data_copy_df, all_types_drug_use)
    marijuana_use_flag = build_group_flag(data_copy_df, substance_marijuana)
    alcohol_use_flag = build_group_flag(data_copy_df, substance_alcohol)
    stimulant_use_flag = build_group_flag(data_copy_df, substance_stimulant)
    nicotine_use_flag = build_group_flag(data_copy_df, substance_nicotine)
    psychoactive_use_flag = build_group_flag(data_copy_df, subtance_psychoactive)
    substance_drugs_use_flag = build_group_flag(data_copy_df, substance_drugs)

    data_copy_df.drop(labels=substances_of_interest, axis=1, inplace=True)
    data_copy_df["opioids-opiates"] = opiods_opiates_1_use
    data_copy_df["all_types_drug_use"] = all_types_drug_use_flag
    data_copy_df["substance_marijuana"] = marijuana_use_flag
    data_copy_df["substance_alcohol"] = alcohol_use_flag
    data_copy_df["substance_stimulant"] = stimulant_use_flag
    data_copy_df["substance_nicotine"] = nicotine_use_flag
    data_copy_df["substance_psychoactive"] = psychoactive_use_flag
    data_copy_df["substance_drugs"] = substance_drugs_use_flag

    return(data_copy_df)


def one_hot_encode(df, cols2encode, missing_label="__MISSING__"):
    """
    One-hot encode `cols2encode`, dropping the first category
    per column. Converts category values to strings to ensure uniform types.
    """
    # copy, fill missing, and coerce to string (uniform dtype)
    df_enc = df[cols2encode].fillna(missing_label).astype(str)

    # sklearn compatibility for sparse vs sparse_output parameter
    try:
        enc = OneHotEncoder(drop='first', sparse=False, handle_unknown="ignore", dtype=int)
    except TypeError:
        enc = OneHotEncoder(drop='first', sparse_output=False, handle_unknown="ignore", dtype=int)

    enc.fit(df_enc)
    one_hot_encoded_df = pd.DataFrame(enc.transform(df_enc),
                                      columns=enc.get_feature_names_out(),
                                      index=df.index)
    return one_hot_encoded_df


### Create minimum analysis data: Basic
with race, age, and gender.

In [196]:
import sys, subprocess
try:
    import pyarrow
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow", "-q"])

# Load AMOS (Automatic MOS) scores for each recording session.
# File: /edata/changye/words_fail/data/merged_amos.parquet
# Column: ablated_amos -> renamed to AMOS
# Join key: 'file' column matches the recording-level index of the analysis DataFrames
amos_path = "/edata/changye/words_fail/data/merged_amos.parquet"
amos_df = (
    pd.read_parquet(amos_path)[["file", "ablated_amos"]]
    .rename(columns={"ablated_amos": "AMOS"})
    .set_index("file")
)
print(f"Loaded AMOS data: {len(amos_df)} recordings, {amos_df['AMOS'].isna().sum()} missing values")
print(amos_df["AMOS"].describe())


Loaded AMOS data: 3098 recordings, 0 missing values
count    3098.000000
mean        3.293043
std         0.963188
min         0.000000
25%         2.612126
50%         3.493932
75%         4.084477
max         4.753661
Name: AMOS, dtype: float64


In [197]:
cols2encode=["race", "gender", "binned_age"]

# X_temp = X_temp.replace(binmappers)# Mapping education to higher level categories

# X_temp.rename(columns={"education":"education_binned"},inplace=True)

minimum_analysis_df = replaced_age_with_binning(main_analysis_df)

minimum_analysis_df = minimum_analysis_df.loc[:, cols2encode + ["pid"]]

# When dropping Unknown values Keep race=999 (Unknown), gender=5 (Other), and binned_age=0 (Unknown) as the reference categories.
# problem with droppping unknown values is small n as reference group, 
# # So instead trying dropping the largest category as the reference group,  Largest categories below:   
#      "race": "1.0",             # White
#     "gender": "2.0",           # Male (or 1.0 Female)
#     "binned_age": "2.0",       # Age 30-45
#     "sexuality": "1.0",        # Heterosexual
#     "employment-status": "3.0", # Full-time work
#     "education_binned": "1.0"  # Grade school to HS (already good)

X_one_hot = one_hot_encode_v2(
    df=minimum_analysis_df,
    cols2encode=cols2encode,
    forced_drop={"race": 1, "gender": 2, "binned_age": 2},
)

# Keep original gender for downstream effect-coding sensitivity checks.
X = minimum_analysis_df.drop(["race", "binned_age", "gender"], axis=1)


# Build response columns directly from main_analysis_df so this cell does not depend on prior Y_* variables.
# Force log-scale WER outcome for regression stability/comparability.
if "log_wer" not in main_analysis_df.columns:
    raw_wer_col = next((c for c in ["wer", "WER"] if c in main_analysis_df.columns), None)
    if raw_wer_col is None:
        raise KeyError("Missing required response column: one of ['log_wer', 'wer', 'WER']")
    main_analysis_df["log_wer"] = np.log1p(main_analysis_df[raw_wer_col].clip(lower=0))

wer_col = "log_wer"
coh_col = next((c for c in ["sentCoherenceSentBertCumulativeCentroid"] if c in main_analysis_df.columns), None)

if coh_col is None:
    raise KeyError("Missing required response column: 'sentCoherenceSentBertCumulativeCentroid'")

Y = main_analysis_df[[wer_col, coh_col]].copy()
Y = Y.rename(columns={wer_col: "Y_WER", coh_col: "Y_COH"})

X_min_analysis = pd.concat([X, X_one_hot, Y], axis=1, join="inner", ignore_index=False)

# Join AMOS score on recording-level index (index = recording file name)
X_min_analysis = X_min_analysis.join(amos_df[["AMOS"]], how="left")
print(f"AMOS joined: {X_min_analysis['AMOS'].notna().sum()} / {len(X_min_analysis)} rows matched")

X_min_analysis.to_csv(out_folder + "basic_analysis.csv")

#X_min_analysis


[one_hot_encode] Dropped reference categories: {'race': '1.0', 'gender': '2.0', 'binned_age': '2.0'}
AMOS joined: 2832 / 2832 rows matched


#### Reference Categories for One-Hot Encoded Categorical Variables

Three demographic variables — race, gender, and age — were one-hot encoded prior to regression modeling. To avoid perfect multicollinearity (the dummy variable trap), one category per variable was omitted from the model and serves as the implicit reference group against which all other categories are compared. Coefficient estimates for the included indicator variables are therefore interpreted as the difference in the outcome relative to that reference group, holding all other covariates constant.

**Race** was encoded from a six-category self-report variable. The reference category is **White** (code 1), the largest racial group in the sample. Indicator variables are included for: Black/African American (code 2), Native American (code 4), Asian (code 5), More than one race (code 6), and Unknown/not reported (code 999). All race coefficients represent the expected difference in the outcome compared to White participants.

**Gender** was encoded from a five-category variable. The reference category is **Male** (code 2). Indicator variables are included for: Female (code 1), Transgender male-to-female (code 3), Transgender female-to-male (code 4), and Other (code 5). All gender coefficients represent the expected difference in the outcome compared to Male participants.

**Age** was represented as a binned ordinal variable. The reference category is **Age 30–45** (code 2), the modal age group in the sample. Indicator variables are included for: Age < 30 (code 1), Age 45–65 (code 3), and Age ≥ 65 (code 4). An "Unknown age" category (code 0) was also retained as an indicator variable rather than set as the reference, to preserve interpretability across all age strata.

Selecting the most prevalent category as the reference group for each variable maximizes the effective sample size in each comparison and reduces standard error inflation for the reference cell estimate.

### Create analysis data: Basic+
Everything in Basic with MOS and SNR added

In [198]:
import ast
import numpy as np

# Helper functions for parsing sequences and extracting transcript-length features
def _parse_sequence(value):
    """Parse array-like or string-formatted sequence."""
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        return list(value)
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None

    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None

    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None

    if isinstance(parsed, (list, tuple, np.ndarray, pd.Series)):
        return list(parsed)
    return None


def _sequence_length(value):
    """Count elements in a sequence."""
    seq = _parse_sequence(value)
    return float(len(seq)) if seq is not None else np.nan


def _numeric_sequence_sum(value):
    """Sum numeric values in a sequence."""
    seq = _parse_sequence(value)
    if seq is None:
        return np.nan

    numeric_values = []
    for item in seq:
        try:
            numeric_values.append(float(item))
        except (TypeError, ValueError):
            continue

    if not numeric_values:
        return np.nan
    return float(sum(numeric_values))


def _numeric_sequence_mean(value):
    """Mean of numeric values in a sequence."""
    seq = _parse_sequence(value)
    if seq is None:
        return np.nan

    numeric_values = []
    for item in seq:
        try:
            numeric_values.append(float(item))
        except (TypeError, ValueError):
            continue

    if not numeric_values:
        return np.nan
    return float(np.mean(numeric_values))


def _segment_recording_duration(value):
    """Use the end timestamp of the last Whisper segment as recording duration (seconds)."""
    seq = _parse_sequence(value)
    if seq is None:
        return np.nan

    for seg in reversed(seq):
        if isinstance(seg, dict) and "end" in seg:
            try:
                return float(seg["end"])
            except (TypeError, ValueError):
                continue
    return np.nan


def _segment_word_count(value):
    """Count words across segment text entries."""
    seq = _parse_sequence(value)
    if seq is None:
        return np.nan

    total = 0
    for seg in seq:
        if isinstance(seg, dict):
            text = seg.get("text", "")
            if isinstance(text, str):
                total += len(text.split())

    return float(total) if total > 0 else np.nan


# Create X_basic_plus with snr and pause_proportion
X_basic_plus = pd.concat(
    [X_min_analysis, main_analysis_df.loc[:, ["snr", "pause_proportion"]]],
    axis=1,
    join="inner",
    ignore_index=False,
)

# Add transcript-length features from Whisper segments
if "segments" in main_analysis_df.columns:
    X_basic_plus["segment_count"] = main_analysis_df["segments"].apply(_sequence_length)
    X_basic_plus["recording_duration"] = main_analysis_df["segments"].apply(_segment_recording_duration)
    X_basic_plus["word_count"] = main_analysis_df["segments"].apply(_segment_word_count)

# Speech rate = words per second of recording
if "word_count" in X_basic_plus.columns and "recording_duration" in X_basic_plus.columns:
    X_basic_plus["speech_rate"] = (
        X_basic_plus["word_count"]
        / X_basic_plus["recording_duration"].replace(0, np.nan)
    )

# Pause-derived features
if "time_diffs" in main_analysis_df.columns:
    _total_pause_duration = main_analysis_df["time_diffs"].apply(_numeric_sequence_sum)
    X_basic_plus["mean_pause_duration"] = main_analysis_df["time_diffs"].apply(_numeric_sequence_mean)

    if "segment_count" in X_basic_plus.columns:
        X_basic_plus["pause_duration_per_segment"] = (
            _total_pause_duration
            / X_basic_plus["segment_count"].replace(0, np.nan)
        )

# Safety guard in case total_pause_duration was already present from a previous run
if "total_pause_duration" in X_basic_plus.columns:
    X_basic_plus = X_basic_plus.drop(columns=["total_pause_duration"])

# Drop snr, collinear with AMOS. Use AMOS instead
if "snr" in X_basic_plus.columns:
    X_basic_plus = X_basic_plus.drop(columns=["snr"])

assert X_basic_plus.shape[0] == X_min_analysis.shape[0], "Seem to have lost some columns during the join"
X_basic_plus.to_csv(out_folder + "basic_plus_analysis.csv")

print(f"X_basic_plus shape: {X_basic_plus.shape}")
print(f"segment_count non-null: {X_basic_plus['segment_count'].notna().sum() if 'segment_count' in X_basic_plus.columns else 0}")
print(f"pause_duration_per_segment non-null: {X_basic_plus['pause_duration_per_segment'].notna().sum() if 'pause_duration_per_segment' in X_basic_plus.columns else 0}")
print(f"mean_pause_duration non-null: {X_basic_plus['mean_pause_duration'].notna().sum() if 'mean_pause_duration' in X_basic_plus.columns else 0}")
print(f"recording_duration non-null: {X_basic_plus['recording_duration'].notna().sum() if 'recording_duration' in X_basic_plus.columns else 0}")
print(f"word_count non-null: {X_basic_plus['word_count'].notna().sum() if 'word_count' in X_basic_plus.columns else 0}")
print(f"speech_rate non-null: {X_basic_plus['speech_rate'].notna().sum() if 'speech_rate' in X_basic_plus.columns else 0}")
#X_basic_plus

X_basic_plus shape: (2832, 24)
segment_count non-null: 2832
pause_duration_per_segment non-null: 2812
mean_pause_duration non-null: 2812
recording_duration non-null: 2832
word_count non-null: 2832
speech_rate non-null: 2832


### Create analysis data: Basic+ Clinical
Everything in Basic+ with the additions of phq9 total, hpsvq total, scl global average

In [199]:
import ast
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

TECH_SCREEN_SOURCE_COLS = [
    "AMOS",
    "snr",
    "pause_proportion",
    "pred_mos",
    "avg_ppl",
    "number_of_nodes",
    "number_of_edges",
    "PE",
    "LSC",
    "density",
    "degree_average",
    "degree_std",
    "L1",
]


def _parse_sequence(value):
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        return list(value)
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None

    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None

    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None

    if isinstance(parsed, (list, tuple, np.ndarray, pd.Series)):
        return list(parsed)
    return None


def _sequence_length(value):
    seq = _parse_sequence(value)
    return float(len(seq)) if seq is not None else np.nan


def _numeric_sequence_summary(value, reducer):
    seq = _parse_sequence(value)
    if seq is None:
        return np.nan

    numeric_values = []
    for item in seq:
        try:
            numeric_values.append(float(item))
        except (TypeError, ValueError):
            continue

    if not numeric_values:
        return np.nan
    return float(reducer(numeric_values))


tech_feature_block = pd.DataFrame(index=main_analysis_df.index)

for col in TECH_SCREEN_SOURCE_COLS:
    if col in main_analysis_df.columns:
        tech_feature_block[col] = pd.to_numeric(main_analysis_df[col], errors="coerce")

if "segments" in main_analysis_df.columns:
    tech_feature_block["segment_count"] = main_analysis_df["segments"].apply(_sequence_length)

if "time_diffs" in main_analysis_df.columns:
    tech_feature_block["pause_event_count"] = main_analysis_df["time_diffs"].apply(_sequence_length)
    tech_feature_block["total_pause_duration"] = main_analysis_df["time_diffs"].apply(
        lambda value: _numeric_sequence_summary(value, sum)
    )
    tech_feature_block["mean_pause_duration"] = main_analysis_df["time_diffs"].apply(
        lambda value: _numeric_sequence_summary(value, np.mean)
    )

tech_feature_block = tech_feature_block.loc[:, ~tech_feature_block.columns.duplicated()].copy()
tech_feature_block = tech_feature_block.apply(pd.to_numeric, errors="coerce")

screen_counts = tech_feature_block.notna().sum().sort_values(ascending=False)
print("Tech-screen candidates and non-null counts:")
print(screen_counts.to_string())

screen_feature_cols = [
    col for col in tech_feature_block.columns
    if tech_feature_block[col].notna().sum() > 0 and tech_feature_block[col].nunique(dropna=True) > 1
]
tech_feature_block = tech_feature_block.loc[:, screen_feature_cols]

for base_name in [
    "X_basic_plus",
    "X_basic_plus_clin",
    "X_basic_plus_clin_sdh",
    "X_location_encoded",
    "X_stratified",
]:
    if base_name in globals() and isinstance(globals()[base_name], pd.DataFrame):
        globals()[f"{base_name}_tech_screen"] = globals()[base_name].join(
            tech_feature_block.drop(columns=[c for c in tech_feature_block.columns if c in globals()[base_name].columns], errors="ignore"),
            how="left"
        )

encoded_base_path = out_folder + "X_basic_plus_clin_sdh_location_encoded.csv"
encoded_screen_path = out_folder + "X_basic_plus_clin_sdh_location_encoded_tech_screen.csv"
X_location_encoded_base = pd.read_csv(encoded_base_path, index_col=0)

if len(X_location_encoded_base) != len(tech_feature_block):
    raise ValueError(
        f"Row count mismatch between encoded base ({len(X_location_encoded_base)}) and tech block ({len(tech_feature_block)})"
    )

alignment_checks = {}
for col in ["snr", "pause_proportion"]:
    if col in X_location_encoded_base.columns and col in tech_feature_block.columns:
        base_vals = pd.to_numeric(X_location_encoded_base[col], errors="coerce").to_numpy()
        tech_vals = tech_feature_block[col].to_numpy()
        mask = np.isfinite(base_vals) & np.isfinite(tech_vals)
        if mask.any():
            alignment_checks[col] = float(np.mean(np.isclose(base_vals[mask], tech_vals[mask], equal_nan=True)))

print("Positional alignment checks:")
print(alignment_checks)

if alignment_checks and min(alignment_checks.values()) < 0.99:
    raise ValueError(f"Positional alignment check failed: {alignment_checks}")

X_location_encoded_tech_screen = X_location_encoded_base.copy()
new_screen_cols = [c for c in tech_feature_block.columns if c not in X_location_encoded_tech_screen.columns]
for col in new_screen_cols:
    X_location_encoded_tech_screen[col] = tech_feature_block[col].to_numpy()

X_location_encoded_tech_screen.to_csv(encoded_screen_path)
print(f"Saved augmented screening dataframe -> {encoded_screen_path}")
print(f"X_location_encoded_tech_screen shape: {X_location_encoded_tech_screen.shape}")
print("Added tech-screen columns and non-null counts in encoded dataframe:")
print(X_location_encoded_tech_screen.loc[:, new_screen_cols].notna().sum().to_string())

wer_candidates = ["Y_WER", "log_wer", "WER", "wer"]
wer_col = next((col for col in wer_candidates if col in X_location_encoded_tech_screen.columns), None)
if wer_col is None:
    raise KeyError("No WER-like outcome column found in tech screening dataframe")
if "pid" not in X_location_encoded_tech_screen.columns:
    raise KeyError("Expected 'pid' column for clustered standard errors")

tech_only_predictors = [
    col for col in tech_feature_block.columns
    if col in X_location_encoded_tech_screen.columns and col != wer_col
]

rows = []
skipped = []
for predictor in tech_only_predictors:
    reg_df = X_location_encoded_tech_screen[[wer_col, "pid", predictor]].dropna().copy()
    if reg_df.empty or reg_df[predictor].nunique(dropna=True) <= 1:
        skipped.append((predictor, "constant_or_empty"))
        continue

    try:
        model = smf.ols(formula=f'Q("{wer_col}") ~ Q("{predictor}")', data=reg_df).fit(
            cov_type="cluster",
            cov_kwds={"groups": reg_df["pid"], "use_correction": True},
        )
        term = f'Q("{predictor}")'
        if term not in model.params.index:
            skipped.append((predictor, "term_missing"))
            continue

        rows.append({
            "Variable": predictor,
            "Estimate": float(model.params[term]),
            "Std. Error": float(model.bse[term]),
            "P>|t|": float(model.pvalues[term]),
            "N": int(len(reg_df)),
        })
    except Exception as exc:
        skipped.append((predictor, str(exc)))

tech_screen_results = pd.DataFrame(rows)
if tech_screen_results.empty:
    raise RuntimeError(f"No tech-screen regressions completed. Skipped: {skipped}")

_, q_values, _, _ = multipletests(tech_screen_results["P>|t|"], method="fdr_bh")
tech_screen_results["q_value"] = q_values
tech_screen_results["fdr_significant"] = tech_screen_results["q_value"] < 0.05
tech_screen_results = tech_screen_results.sort_values(["q_value", "P>|t|"]).reset_index(drop=True)

tech_screen_output_path = out_folder + "Univariate_Regression_from_X_basic_plus_clin_sdh_location_encoded_tech_screen.csv"
tech_screen_results.to_csv(tech_screen_output_path, index=False)

print(f"Saved tech-screen univariate results -> {tech_screen_output_path}")
print("\nTop tech-screen hits:")
print(tech_screen_results.head(15).to_string(index=False))
if skipped:
    print("\nSkipped predictors:")
    for predictor, reason in skipped:
        print(f" - {predictor}: {reason}")

Tech-screen candidates and non-null counts:
snr                     2832
pause_proportion        2832
pred_mos                2832
avg_ppl                 2832
number_of_nodes         2832
number_of_edges         2832
PE                      2832
LSC                     2832
density                 2832
degree_average          2832
degree_std              2832
L1                      2832
segment_count           2832
pause_event_count       2832
total_pause_duration    2812
mean_pause_duration     2812


Positional alignment checks:
{'snr': 1.0, 'pause_proportion': 1.0}
Saved augmented screening dataframe -> /edata/obdw/sandwich_analysis_data/X_basic_plus_clin_sdh_location_encoded_tech_screen.csv
X_location_encoded_tech_screen shape: (2832, 96)
Added tech-screen columns and non-null counts in encoded dataframe:
pred_mos             2832
avg_ppl              2832
number_of_nodes      2832
number_of_edges      2832
PE                   2832
LSC                  2832
density              2832
degree_average       2832
degree_std           2832
L1                   2832
pause_event_count    2832
Saved tech-screen univariate results -> /edata/obdw/sandwich_analysis_data/Univariate_Regression_from_X_basic_plus_clin_sdh_location_encoded_tech_screen.csv

Top tech-screen hits:
            Variable  Estimate  Std. Error        P>|t|    N      q_value  fdr_significant
                 snr -0.042148    0.003919 5.650154e-27 2832 9.040246e-26             True
            pred_mos -0.270899    0.043

In [200]:
# Keep the relevant clinical variables, include suicidal thoughts, and one-hot encode diagnosis fields when needed.

clinical_base = ["phq9-total", 
                 "hpsvq-total-score", 
               
                 # HPSVQ items related to the characteristics of voices, ""hpsvq-voice-score""
                 "hpsvq-1-frequency",
                 "hpsvq-3-volume-of-voices",
                 "hpsvq-4-voices-length",
                 "hpsvq-8-clarity-of-voices",
                 "hpsvq-10-time-of-day-of-voices",
                 "hpsvq-12-where-are-the-voices",
                
                # HPSVQ items related to the distress of voices
                 "hpsvq-2-bad-voices",
                 "hpsvq-5-interference-in-activities",
                 "hpsvq-6-distressing-voices",
                 "hpsvq-7-worthless-useless-voices",
                 "hpsvq-9-follow-voices-orders",
                 "hpsvq-11-social-situations",
                 "hpsvq-13-typical-week",

                "scl-avg-global-score",
                'sds-1-work-school-disruption', 'sds-2-social-leisure-disruption', 'sds-3-home-family-disruption', 
                'sds-4-school-work-missed', 'sds-5-less-productive-days', 'sds-6-2t-worked-for-other-reasons'
 ]
suicidal_candidates = [
    "phq9-9-suicidal-thoughts",
    "suicidal-thoughts",
    "phq9-suicidal-thoughts",
]

# Diagnosis columns to include if present in the data
diagnosis_candidates = [
    #"diagnoses",
    "dx-alzheimers-parkinsons",
    "dx-bipolar-disorder",
    "dx-depression",
    "dx-tbi",
    "dx-migraines",
    "dx-schizoaffective-disorder",
    "dx-schizophrenia",
    "dx-ptsd",
    "dx-substance-use",
    "dx-seizures",
    "dx-none-of-the-above",
    #"dx-other",
]


suicidal_col = next((c for c in suicidal_candidates if c in main_analysis_df.columns), None)
diagnosis_existing = [c for c in diagnosis_candidates if c in main_analysis_df.columns]

# Base clinical columns + suicidal column (if found)
clinical_cols = [c for c in clinical_base if c in main_analysis_df.columns]
if suicidal_col is not None:
    clinical_cols.append(suicidal_col)

clinical_df = main_analysis_df.loc[:, list(dict.fromkeys(clinical_cols))].copy()

# Split diagnosis columns into numeric/binary vs object/categorical for one-hot encoding
diag_obj_cols = [
    c for c in diagnosis_existing
    if (main_analysis_df[c].dtype == "object" or str(main_analysis_df[c].dtype).startswith("category"))
]
diag_num_cols = [c for c in diagnosis_existing if c not in diag_obj_cols]

diag_num_df = main_analysis_df.loc[:, diag_num_cols].copy() if diag_num_cols else pd.DataFrame(index=main_analysis_df.index)
diag_ohe_df = pd.DataFrame(index=main_analysis_df.index)

if diag_obj_cols:
    diag_ohe_df = one_hot_encode(df=main_analysis_df, cols2encode=diag_obj_cols)
    diag_ohe_df.columns = [f"diag_{c}" for c in diag_ohe_df.columns]

X_basic_plus_clin = pd.concat(
    [X_basic_plus, clinical_df, diag_num_df, diag_ohe_df],
    axis=1,
    join="inner",
    ignore_index=False
)


# Group diagnosis columns into clinically meaningful bins
dx_groups = {
    "dx_group_smi": [  # Serious Mental Illness
        "dx-bipolar-disorder",
        "dx-schizoaffective-disorder",
        "dx-schizophrenia",
        "dx-depression",
    ],
    "dx_group_substance": [  # Substance use disorder
        "dx-substance-use",
    ],
    "dx_group_ptsd": [  # PTSD
        "dx-ptsd",
    ],
    "dx_group_neuro_med": [  # Neurological / Medical
        "dx-alzheimers-parkinsons",
        "dx-migraines",
        "dx-seizures",
        "dx-tbi",
    ],
}

def _to_binary_dx(s: pd.Series) -> pd.Series:
    # Handles numeric/bool/object diagnosis encodings robustly
    if pd.api.types.is_numeric_dtype(s):
        return (s.fillna(0) > 0).astype(float)

    s2 = s.astype(str).str.strip().str.lower()
    true_vals = {"1", "1.0", "true", "t", "yes", "y"}
    return s2.isin(true_vals).astype(float)

# Build grouped diagnosis features
dx_group_df = pd.DataFrame(index=main_analysis_df.index)

for group_name, cols in dx_groups.items():
    existing = [c for c in cols if c in main_analysis_df.columns]
    if existing:
        bin_df = pd.concat([_to_binary_dx(main_analysis_df[c]) for c in existing], axis=1)
        dx_group_df[group_name] = bin_df.max(axis=1).astype(float)
    else:
        dx_group_df[group_name] = 0.0

# Optional: keep explicit none-of-the-above signal (if available)
if "dx-none-of-the-above" in main_analysis_df.columns:
    dx_group_df["dx_group_none"] = _to_binary_dx(main_analysis_df["dx-none-of-the-above"])
print("Diagnosis group columns created:", dx_group_df.columns.tolist())
print(dx_group_df.describe(include="all"))

# Use grouped bins instead of individual diagnosis columns
X_basic_plus_clin = pd.concat(
    [X_basic_plus, clinical_df, dx_group_df],
    axis=1,
    join="inner",
    ignore_index=False
)
X_basic_plus_clin = X_basic_plus_clin.loc[:, ~X_basic_plus_clin.columns.duplicated()]

# Drop duplicate column names if any appear from merges
X_basic_plus_clin = X_basic_plus_clin.loc[:, ~X_basic_plus_clin.columns.duplicated()]

assert X_basic_plus_clin.shape[0] == X_basic_plus.shape[0], "Seem to have lost some rows during the join"
if suicidal_col is not None:
    assert suicidal_col in X_basic_plus_clin.columns

# Add HPSVQ subtotal scores from the underlying item responses
hpsvq_voice_components = [
    "hpsvq-1-frequency",
    "hpsvq-3-volume-of-voices",
    "hpsvq-4-voices-length",
    "hpsvq-8-clarity-of-voices",
    "hpsvq-10-time-of-day-of-voices",
    "hpsvq-12-where-are-the-voices",
]
hpsvq_distress_components = [
    "hpsvq-2-bad-voices",
    "hpsvq-5-interference-in-activities",
    "hpsvq-6-distressing-voices",
    "hpsvq-7-worthless-useless-voices",
    "hpsvq-9-follow-voices-orders",
    "hpsvq-11-social-situations",
    "hpsvq-13-typical-week",
]
hpsvq_items_to_drop = []

available_hpsvq_voice_components = [c for c in hpsvq_voice_components if c in X_basic_plus_clin.columns]
if available_hpsvq_voice_components:
    X_basic_plus_clin["hpsvq-voice-score"] = X_basic_plus_clin[available_hpsvq_voice_components].sum(axis=1, min_count=1)
    hpsvq_items_to_drop.extend(available_hpsvq_voice_components)
    print(f"Created hpsvq-voice-score from: {available_hpsvq_voice_components}")
else:
    print("WARNING: No HPSVQ voice characteristic components found; hpsvq-voice-score not created")

available_hpsvq_distress_components = [c for c in hpsvq_distress_components if c in X_basic_plus_clin.columns]
if available_hpsvq_distress_components:
    X_basic_plus_clin["hpsvq-distress-score"] = X_basic_plus_clin[available_hpsvq_distress_components].sum(axis=1, min_count=1)
    hpsvq_items_to_drop.extend(available_hpsvq_distress_components)
    print(f"Created hpsvq-distress-score from: {available_hpsvq_distress_components}")
else:
    print("WARNING: No HPSVQ distress components found; hpsvq-distress-score not created")

# Drop the individual HPSVQ items only after their subtotal scores have been created.
if hpsvq_items_to_drop:
    hpsvq_items_to_drop = list(dict.fromkeys(hpsvq_items_to_drop))
    X_basic_plus_clin = X_basic_plus_clin.drop(columns=hpsvq_items_to_drop, errors="ignore")
    print(f"Dropped individual HPSVQ item columns after subtotal creation: {hpsvq_items_to_drop}")

print("HPSVQ subtotal columns now present:", [
    c for c in ["hpsvq-voice-score", "hpsvq-distress-score"] if c in X_basic_plus_clin.columns
])

# Add Sheehan Disability Scale total from the 3 core disruption items
sds_total_components = [
    'sds-1-work-school-disruption',
    'sds-2-social-leisure-disruption',
    'sds-3-home-family-disruption'
 ]
available_sds_components = [c for c in sds_total_components if c in X_basic_plus_clin.columns]
if available_sds_components:
    X_basic_plus_clin["SDS_Total"] = X_basic_plus_clin[available_sds_components].sum(axis=1, min_count=1)
    print(f"Created SDS_Total from: {available_sds_components}")
else:
    print("WARNING: No SDS disruption components found; SDS_Total not created")

# Clinically interpretable binary variables
# PHQ9_high = PHQ9 >= 10
# SCL9_moderate = SCL9 >= 1.0 (cutoff for moderate level of distress compared to normal population)
# SCL9_High = SCL9 >= 1.7 (cutoff for high level of distress)
# AVH_distress_high = distress >= 4
X_basic_plus_clin["PHQ9_high"] = (X_basic_plus_clin["phq9-total"] >= 10).astype(float)
X_basic_plus_clin["SCL9_moderate"] = (X_basic_plus_clin["scl-avg-global-score"] >= 1.0).astype(float)
X_basic_plus_clin["SCL9_High"] = (X_basic_plus_clin["scl-avg-global-score"] >= 1.7).astype(float)
X_basic_plus_clin["SDS_High"] = (X_basic_plus_clin["SDS_Total"] >= 21).astype(float) if "SDS_Total" in X_basic_plus_clin.columns else np.nan

print("Clinical binary columns created:")
print(X_basic_plus_clin[["PHQ9_high"]].describe(include="all"))
if "SDS_Total" in X_basic_plus_clin.columns:
    print("SDS_Total summary:")
    print(X_basic_plus_clin[["SDS_Total"]].describe(include="all"))

print("Clinical columns included:", clinical_df.columns.tolist())
print("Diagnosis numeric columns included:", diag_num_cols)
print("Diagnosis object columns one-hot encoded:", diag_obj_cols)
print(X_basic_plus_clin.columns.tolist())

X_basic_plus_clin.to_csv(out_folder + "basic_plus_clinical_analysis.csv")

Diagnosis group columns created: ['dx_group_smi', 'dx_group_substance', 'dx_group_ptsd', 'dx_group_neuro_med', 'dx_group_none']
       dx_group_smi  dx_group_substance  dx_group_ptsd  dx_group_neuro_med  \
count   2832.000000         2832.000000    2832.000000         2832.000000   
mean       0.938912            0.211158       0.326624            0.271893   
std        0.239533            0.408202       0.469061            0.445013   
min        0.000000            0.000000       0.000000            0.000000   
25%        1.000000            0.000000       0.000000            0.000000   
50%        1.000000            0.000000       0.000000            0.000000   
75%        1.000000            0.000000       1.000000            1.000000   
max        1.000000            1.000000       1.000000            1.000000   

       dx_group_none  
count    2832.000000  
mean        0.043432  
std         0.203864  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%     

In [201]:
def one_hot_encode(df, cols2encode, missing_label="__MISSING__", forced_drop=None, prefer_drop=("other", "unknown"), fallback="most_frequent", verbose=True):
    """One-hot encode columns while explicitly selecting a reference category to drop per column."""
    df_enc = df[cols2encode].fillna(missing_label).astype(str)
    forced_drop = forced_drop or {}

    def _resolve_requested_category(requested, categories):
        requested_norm = str(requested).strip().lower()
        for cat in categories:
            if cat.strip().lower() == requested_norm:
                return cat

        try:
            requested_num = float(requested)
        except (TypeError, ValueError):
            requested_num = None

        if requested_num is not None:
            for cat in categories:
                try:
                    if float(cat) == requested_num:
                        return cat
                except (TypeError, ValueError):
                    continue
            return None

        for cat in categories:
            if requested_norm in cat.strip().lower():
                return cat
        return None

    drop_values = []
    dropped_map = {}

    for col in cols2encode:
        value_counts = df_enc[col].value_counts()
        categories = list(value_counts.index.astype(str))
        chosen = None

        if col in forced_drop:
            chosen = _resolve_requested_category(forced_drop[col], categories)
            if chosen is None:
                print(f"[one_hot_encode] Requested drop '{forced_drop[col]}' for '{col}' was not found. Falling back to automatic rule.")

        if chosen is None:
            for token in prefer_drop:
                chosen = _resolve_requested_category(token, categories)
                if chosen is not None:
                    break

        if chosen is None:
            if fallback == "least_frequent":
                chosen = str(value_counts.idxmin())
            else:
                chosen = str(value_counts.idxmax())

        drop_values.append(chosen)
        dropped_map[col] = chosen

    if verbose:
        print(f"[one_hot_encode] Dropped reference categories: {dropped_map}")

    try:
        enc = OneHotEncoder(drop=drop_values, sparse=False, handle_unknown="ignore", dtype=int)
    except TypeError:
        enc = OneHotEncoder(drop=drop_values, sparse_output=False, handle_unknown="ignore", dtype=int)

    enc.fit(df_enc)
    one_hot_encoded_df = pd.DataFrame(
        enc.transform(df_enc),
        columns=enc.get_feature_names_out(),
        index=df.index
    )
    return one_hot_encoded_df


### Create analysis data: Basic+ Clinical and SDH
Everything in Basic+ Clinical model with the additions of SDH features (sexuality, employment-status, education, and all drug data)

In [202]:
# Get whether the user has used any drugs, and whether they have used opioids or opiates. This is important to include in the analysis since substance use is a key factor that can impact both WER and Coherence, and we want to make sure we are accounting for it in our models.
substance_feature_cols = [
    "opioids-opiates",
    "all_types_drug_use",
    "substance_marijuana",
    "substance_alcohol",
    "substance_stimulant",
    "substance_nicotine",
    "substance_psychoactive",
]
opiod_data = bin_substance_use(data_df=main_analysis_df)
available_substance_feature_cols = [c for c in substance_feature_cols if c in opiod_data.columns]
opiod_data = opiod_data.loc[:, available_substance_feature_cols]
temp = main_analysis_df.copy()
temp = temp.merge(opiod_data, left_index=True, right_index=True, how='left')
print("List of temporary columns after merging substance use data:")
print(list(temp.columns))

# Quick sanity-check prevalence before encoding
print("\nSubstance grouped flag prevalence (value counts):")
for col in available_substance_feature_cols:
    print(f"\n{col}:")
    print(temp[col].value_counts(dropna=False).sort_index())

temp = temp.replace(binmappers)# Mapping education to higher level categories
temp.rename(columns={"education":"education_binned"},inplace=True)
# Change which columns to encode: only true categoricals, NOT binary substance flags
# Binary flags (0/1) are already encoded and should not be one-hot encoded
# OneHot encoding - dropping the largest category as reference group to avoid small n issues with unknown/other categories. For example, for sexuality, dropping "heterosexual" as the reference group since it's the largest category, and keeping
    # "sexuality": "1.0",        # Heterosexual
    # "employment-status": "3.0", # Full-time work
    # "education_binned": "1.0"  # Grade school to HS (already good)

cols2encode=["sexuality", "employment-status", "education_binned"]
X_one_hot = one_hot_encode(
    df=temp,
    cols2encode=cols2encode,
    forced_drop={"sexuality": 1, "employment-status": 3, "education_binned": "1.0"},
)
# Add binary substance flags directly without one-hot encoding (they're already 0/1)
X_basic_plus_clin_sdh = pd.concat([X_basic_plus_clin, X_one_hot, opiod_data], axis=1, join="inner", ignore_index=False)
print("Columns after one-hot encoding:")
print(list(X_basic_plus_clin_sdh.columns))

assert X_basic_plus_clin_sdh.shape[0] == X_basic_plus_clin.shape[0], "Seem to have lost some columns during the join"
X_basic_plus_clin_sdh.to_csv(out_folder + "basic_plus_clinical_sdh_analysis.csv")
#X_basic_plus_clin_sdh


List of temporary columns after merging substance use data:
['pid', 'record', 'transcription', 'audio', 'prediction', 'wer', 'cer', 'record_date', 'in_person', 'days of data', 'audio diary entries', 'age', 'gender', 'phone', 'data_plan', 'ah_freq', 'location', 'availability', 'future_contact', 'comp_q1', 'comp_q2', 'comp_q3', 'comp_q4', 'comp_q5', 'race', 'hispanic', 'marital-status', 'sexuality', 'education', 'employment-status', 'living-status', 'basic-cell', 'smartphone', 'tablet', 'computer', 'wearable-tech', 'email', 'social-media', 'verify-hearing-voices', 'how-often', 'how-long', 'diagnoses', 'dx-alzheimers-parkinsons', 'dx-bipolar-disorder', 'dx-depression', 'dx-tbi', 'dx-migraines', 'dx-schizoaffective-disorder', 'dx-schizophrenia', 'dx-ptsd', 'dx-substance-use', 'dx-seizures', 'dx-none-of-the-above', 'dx-other', 'used-treatment', 'type-of-treatments', 'tx-one-on-one', 'tx-online', 'tx-group', 'tx-partial-hospitalization', 'tx-alcohol-drug-rehab', 'tx-telepsychiatry', 'tx-res-

In [203]:
# Comparison cell for one-hot encoder.
# Gender is coded numerically in this notebook: other=5, male=2.
_min_cols2encode = ["race", "gender", "binned_age"]

X_one_hot_gender_other = one_hot_encode(
    df=minimum_analysis_df,
    cols2encode=_min_cols2encode,
    forced_drop={"race": 999, "gender": 5, "binned_age": 0},
    verbose=True,
)

X_one_hot_gender_male = one_hot_encode(
    df=minimum_analysis_df,
    cols2encode=_min_cols2encode,
    forced_drop={"race": 999, "gender": 2, "binned_age": 0},
    verbose=True,
)

gender_cols_other = sorted([c for c in X_one_hot_gender_other.columns if c.startswith("gender_")])
gender_cols_male = sorted([c for c in X_one_hot_gender_male.columns if c.startswith("gender_")])

print("Gender dummies when dropping gender=5 (other):", gender_cols_other)
print("Gender dummies when dropping gender=2 (male):", gender_cols_male)
print("Only in other-drop version:", sorted(set(gender_cols_other) - set(gender_cols_male)))
print("Only in male-drop version:", sorted(set(gender_cols_male) - set(gender_cols_other)))


[one_hot_encode] Dropped reference categories: {'race': '999.0', 'gender': '5.0', 'binned_age': '0.0'}
[one_hot_encode] Dropped reference categories: {'race': '999.0', 'gender': '2.0', 'binned_age': '0.0'}
Gender dummies when dropping gender=5 (other): ['gender_1.0', 'gender_2.0', 'gender_3.0', 'gender_4.0']
Gender dummies when dropping gender=2 (male): ['gender_1.0', 'gender_3.0', 'gender_4.0', 'gender_5.0']
Only in other-drop version: ['gender_2.0']
Only in male-drop version: ['gender_5.0']


In [204]:
# Summary statistics for X_basic_plus_clin_sdh
print("=" * 80)
print("SUMMARY STATISTICS: X_basic_plus_clin_sdh")
print("=" * 80)

print("\n1. DATASET SHAPE AND SIZE:")
print(f"   - Rows: {X_basic_plus_clin_sdh.shape[0]}")
print(f"   - Columns: {X_basic_plus_clin_sdh.shape[1]}")
print(f"   - Total cells: {X_basic_plus_clin_sdh.shape[0] * X_basic_plus_clin_sdh.shape[1]}")

print("\n2. DATA TYPES:")
print(X_basic_plus_clin_sdh.dtypes.value_counts())

print("\n3. MISSING VALUES:")
missing_counts = X_basic_plus_clin_sdh.isna().sum()
if missing_counts.sum() == 0:
    print("   - No missing values")
else:
    print(missing_counts[missing_counts > 0])

print("\n4. DESCRIPTIVE STATISTICS:")
print(X_basic_plus_clin_sdh.describe().T)

# Correlations: resolve outcome names robustly after column renames.
corr_df = X_basic_plus_clin_sdh.corr(numeric_only=True)
wer_candidates = ["WER", "wer", "log_wer", "Y_WER"]
coh_candidates = ["sentCoherenceSentBertCumulativeCentroid", "Y_COH"]
wer_col = next((c for c in wer_candidates if c in corr_df.columns), None)
coh_col = next((c for c in coh_candidates if c in corr_df.columns), None)

print("\n5. CORRELATIONS WITH RESPONSE VARIABLES:")
if wer_col is not None:
    print(f"   - Correlation with {wer_col}:")
    print(corr_df[wer_col].sort_values(ascending=False).head(10))
else:
    print("   - WER correlation skipped: no WER-like column found.")

if coh_col is not None:
    print(f"\n   - Correlation with {coh_col}:")
    print(corr_df[coh_col].sort_values(ascending=False).head(10))
else:
    print("\n   - Coherence correlation skipped: no coherence-like column found.")

print("\n6. MEMORY USAGE:")
print(f"   - Total memory: {X_basic_plus_clin_sdh.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "=" * 80)

SUMMARY STATISTICS: X_basic_plus_clin_sdh

1. DATASET SHAPE AND SIZE:
   - Rows: 2832
   - Columns: 62
   - Total cells: 175584

2. DATA TYPES:
float64    39
int64      23
Name: count, dtype: int64

3. MISSING VALUES:
mean_pause_duration           20
pause_duration_per_segment    20
dtype: int64

4. DESCRIPTIVE STATISTICS:
                         count         mean         std  min    25%     50%  \
pid                     2832.0  1310.259534  696.878051  3.0  677.0  1441.0   
race_2.0                2832.0     0.167373    0.373374  0.0    0.0     0.0   
race_4.0                2832.0     0.005297    0.072598  0.0    0.0     0.0   
race_5.0                2832.0     0.037076    0.188982  0.0    0.0     0.0   
race_6.0                2832.0     0.132415    0.339002  0.0    0.0     0.0   
...                        ...          ...         ...  ...    ...     ...   
substance_marijuana     2832.0     0.244350    0.429777  0.0    0.0     0.0   
substance_alcohol       2832.0     0.267302

In [205]:
# --- Merge Geographic Location Data into X_basic_plus_clin_sdh ---
import pandas as pd
import os

# Load the geographic location data
location_path = '/home/NETID/obdw4/location_data_patrick/olivermergedcounts.csv'
try:
    location_df = pd.read_csv(location_path)
    print(f"Loaded location data: {location_df.shape}")
    print(f"Location data columns: {list(location_df.columns)}")
    
    # Filter to primary cluster only
    if 'is_primary_cluster' in location_df.columns:
        location_df_primary = location_df[location_df['is_primary_cluster'] == True].copy()
        print(f"Filtered to primary clusters only: {location_df_primary.shape}")
        print(f"Unique participants in location data: {location_df_primary['account_id'].nunique()}")
    else:
        print("WARNING: 'is_primary_cluster' column not found, using all location data")
        location_df_primary = location_df.copy()
    
    # Select only essential geographic columns
    # Keep: account_id (for merge), SVI indices (RPL_THEMES + subscores), RUCA codes, and STATEFP
    # ToDo: for now removing state as it's too many variables, do another one with state, or region
    essential_columns = [
        'account_id',           # For merging with pid
        'RPL_THEMES',           # Overall SVI percentile
        'RPL_THEME1',           # Socioeconomic SVI subscore
        'RPL_THEME2',           # Household composition SVI subscore
        'RPL_THEME3',           # Minority status/language SVI subscore
        'RPL_THEME4',           # Housing/transportation SVI subscore
        'PrimaryRUCA',          # Urban/rural classification (primary)
        # 'SecondaryRUCA',        # Urban/rural classification (secondary) , removed Secondary           
        #'STATEFP',               # State FIPS code
        'doctor_count',         # Number of doctors/medical facilities in the area
        'pharmacy_count',       # Number of pharmacies in the area
        'park_count',           # Number of parks in the area
        'bus_station_count',    # Number of bus stations in the area
        'supermarket_count',    # Number of supermarkets in the area
        'cluster'               # Cluster identifier (geographic cluster assignment)
    ]
    
    # Filter to only columns that exist in the dataframe
    available_columns = [col for col in essential_columns if col in location_df_primary.columns]
    missing_columns = [col for col in essential_columns if col not in location_df_primary.columns]
    
    if missing_columns:
        print(f"\nWARNING: Missing expected columns: {missing_columns}")
    
    print(f"\nSelecting {len(available_columns)} essential geographic columns:")
    print(f"  {available_columns}")
    
    # Keep only essential columns
    location_df_primary = location_df_primary[available_columns]
    print(f"Reduced location data shape: {location_df_primary.shape}")
    
    # Dropped columns summary
    print(f"\nDropped columns:")
    print(f"  - Some of Google Maps count variables (doctor_count, pharmacy_count, etc.)")
    print(f"  - Demographic/clinical duplicates (age, gender, race, clinical scores, etc.)")
    print(f"  - Cluster statistics (cluster_rank, total_minutes, etc.)")
    print(f"  - Raw coordinates (medoid_lat/lon, latitude/longitude)")
    print(f"  - Event tracking data (event_data, event_time)")
    print(f"  - Geographic identifiers (COUNTYFP, TRACTFP, GEOID)")
        
except Exception as e:
    print(f"Could not load location data: {e}")
    location_df_primary = None

# Display columns in X_basic_plus_clin_sdh
print(f"\nX_basic_plus_clin_sdh shape: {X_basic_plus_clin_sdh.shape}")
print(f"Unique participants in X_basic_plus_clin_sdh: {X_basic_plus_clin_sdh['pid'].nunique()}")

# Merge on account_id = pid
if location_df_primary is not None:
    if 'account_id' not in location_df_primary.columns:
        print("ERROR: 'account_id' not found in location data.")
    elif 'pid' not in X_basic_plus_clin_sdh.columns:
        print("ERROR: 'pid' not found in X_basic_plus_clin_sdh.")
    else:
        print(f"\nMerging on: location_df['account_id'] = X_basic_plus_clin_sdh['pid']")
        
        # Merge on PID only (one primary cluster per participant)
        merged_df = X_basic_plus_clin_sdh.merge(
            location_df_primary, 
            left_on='pid', 
            right_on='account_id', 
            how='left'
        )
        
        # Remove duplicate rows from merged_df
        merged_df = merged_df.drop_duplicates()
        
        print(f"\nMerged DataFrame shape: {merged_df.shape}")
        print(f"Original X_basic_plus_clin_sdh shape: {X_basic_plus_clin_sdh.shape}")
        print(f"Rows with location data: {merged_df['account_id'].notna().sum()}")
        print(f"Rows without location data: {merged_df['account_id'].isna().sum()}")
        
        # Save
        merged_df.to_csv(out_folder + "X_basic_plus_clin_sdh_with_location.csv")
        print("\nSaved merged DataFrame with location data.")
else:
    print("Location data not loaded; skipping merge.")

Loaded location data: (1038, 114)
Location data columns: ['medoid_lat', 'medoid_lon', 'cluster', 'account_id', 'doctor_count', 'pharmacy_count', 'gym_count', 'library_count', 'park_count', 'bus_station_count', 'transit_station_count', 'supermarket_count', 'event_data', 'event_time', 'longitude', 'latitude', 'STATEFP', 'COUNTYFP', 'TRACTFP', 'TRACT_GEOID', 'GEOID', 'PrimaryRUCA', 'SecondaryRUCA', 'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4', 'RPL_THEMES', 'in_person', 'account_status', 'page_progress', 'days of data', 'Tracking', 'age', 'gender', 'phone', 'data_plan', 'ah_freq', 'location', 'availability', 'contact', 'comp_q1', 'comp_q2', 'comp_q3', 'comp_q4', 'comp_q5', 'race', 'hispanic', 'marital-status', 'sexuality', 'education', 'employment-status', 'living-status', 'basic-cell', 'smartphone', 'tablet', 'computer', 'wearable-tech', 'email', 'social-media', 'verify-hearing-voices', 'how-often', 'how-long', 'diagnoses', 'used-treatment', 'type-of-treatments', 'who-knows', '

Filtered to primary clusters only: (370, 114)
Unique participants in location data: 370

Selecting 13 essential geographic columns:
  ['account_id', 'RPL_THEMES', 'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4', 'PrimaryRUCA', 'doctor_count', 'pharmacy_count', 'park_count', 'bus_station_count', 'supermarket_count', 'cluster']
Reduced location data shape: (370, 13)

Dropped columns:
  - Some of Google Maps count variables (doctor_count, pharmacy_count, etc.)
  - Demographic/clinical duplicates (age, gender, race, clinical scores, etc.)
  - Cluster statistics (cluster_rank, total_minutes, etc.)
  - Raw coordinates (medoid_lat/lon, latitude/longitude)
  - Event tracking data (event_data, event_time)
  - Geographic identifiers (COUNTYFP, TRACTFP, GEOID)

X_basic_plus_clin_sdh shape: (2832, 62)
Unique participants in X_basic_plus_clin_sdh: 269

Merging on: location_df['account_id'] = X_basic_plus_clin_sdh['pid']

Merged DataFrame shape: (2832, 75)
Original X_basic_plus_clin_sdh shape

#### Understanding the Geographic Cluster Data

**Cluster Information:**
- **`cluster`**: Numeric identifier for geographic clusters where participants spend time
- Each participant can have multiple clusters (different locations they visit)
- **`is_primary_cluster`**: Boolean flag indicating the main location (use this for analysis)
- **`is_secondary_cluster`**: Boolean flag for secondary locations

**Cluster-Level Variables** (already included in merge):
- **Location coordinates**: `medoid_lat`, `medoid_lon` (not included to reduce dimensionality)
- **POI counts**: `doctor_count`, `pharmacy_count`, `hospital_count`, `park_count`, `bus_station_count`, `supermarket_count`
- **SVI (Social Vulnerability Index)**: `RPL_THEMES` (overall), `RPL_THEME1-4` (subscores)
- **RUCA codes**: `PrimaryRUCA`, `SecondaryRUCA` (urban/rural classification)
- **Census geography**: `STATEFP`, `COUNTYFP`, `TRACTFP`, `GEOID` (not included to reduce dimensionality)

**Data Source**: Dr. Patrick Wedgeworth's location clustering analysis

**Note**: Currently only using primary cluster data (`is_primary_cluster == True`) to ensure one geographic location per participant. Each row in the merged data represents a participant's primary geographic cluster.

### Clean and Encode Location Data
Remove duplicate columns from the location merge and one-hot encode categorical geographic variables

In [206]:
# Build/fetch X_location_final from whichever upstream dataframe is available in this session.
if "X_location_final" in locals() and isinstance(X_location_final, pd.DataFrame):
    _location_base = X_location_final.copy()
elif "X_location_encoded_base" in locals() and isinstance(X_location_encoded_base, pd.DataFrame):
    _location_base = X_location_encoded_base.copy()
elif "merged_df" in locals() and isinstance(merged_df, pd.DataFrame):
    _location_base = merged_df.copy()
else:
    raise NameError(
        "X_location_final is not defined and no fallback dataframe is available. "
        "Run the location merge/encoding cell first."
    )

# Remove merge key helper if present; modeling file uses pid.
if "account_id" in _location_base.columns:
    _location_base = _location_base.drop(columns=["account_id"], errors="ignore")

# Align row order to main_analysis_df when indices match but order differs.
if len(_location_base) != len(main_analysis_df):
    if set(main_analysis_df.index).issubset(set(_location_base.index)):
        _location_base = _location_base.reindex(main_analysis_df.index)
    else:
        raise ValueError(
            f"Row count mismatch: location_base={len(_location_base)}, main_analysis_df={len(main_analysis_df)}"
        )

if "segments" not in main_analysis_df.columns or "time_diffs" not in main_analysis_df.columns:
    raise KeyError("Expected 'segments' and 'time_diffs' in main_analysis_df")

segments_array = main_analysis_df["segments"].values
time_diffs_array = main_analysis_df["time_diffs"].values

# Process arrays element-by-element using positional indexing.
segment_counts = [_sequence_length(seg) for seg in segments_array]
recording_durations = [_segment_recording_duration(seg) for seg in segments_array]
word_counts = [_segment_word_count(seg) for seg in segments_array]
total_pause_durations = [_numeric_sequence_sum(td) for td in time_diffs_array]
mean_pause_durations = [_numeric_sequence_mean(td) for td in time_diffs_array]

X_location_final = _location_base.copy()
X_location_final["segment_count"] = segment_counts
X_location_final["recording_duration"] = recording_durations
X_location_final["word_count"] = word_counts
X_location_final["total_pause_duration"] = total_pause_durations
X_location_final["mean_pause_duration"] = mean_pause_durations

# Normalized pause and speech-rate metrics.
X_location_final["pause_duration_per_segment"] = (
    X_location_final["total_pause_duration"]
    / X_location_final["segment_count"].replace(0, np.nan)
)
X_location_final["speech_rate"] = (
    X_location_final["word_count"]
    / X_location_final["recording_duration"].replace(0, np.nan)
)

# Save final dataset.
X_location_final.to_csv(out_folder + "X_basic_plus_clin_sdh_location_encoded.csv")
print("Saved X_basic_plus_clin_sdh_location_encoded.csv")
print(f"Final dataset shape: {X_location_final.shape}")

for col in [
    "segment_count",
    "pause_duration_per_segment",
    "mean_pause_duration",
    "recording_duration",
    "word_count",
    "speech_rate",
]:
    if col in X_location_final.columns:
        print(f"{col} non-null: {X_location_final[col].notna().sum()}")

location_specific_cols = [col for col in X_location_final.columns if col not in X_basic_plus_clin_sdh.columns]
print(f"Location-specific columns: {len(location_specific_cols)}")

Saved X_basic_plus_clin_sdh_location_encoded.csv
Final dataset shape: (2832, 85)
segment_count non-null: 2832
pause_duration_per_segment non-null: 2812
mean_pause_duration non-null: 2812
recording_duration non-null: 2832
word_count non-null: 2832
speech_rate non-null: 2832
Location-specific columns: 23


### Create Stratification Variables
Create meaningful stratification variables for cluster-adjusted analyses:
- **Urban/Rural Categories**: From RUCA codes
- **SVI Vulnerability Levels**: Low/Medium/High based on RPL_THEMES percentiles
- **Healthcare Access**: Based on combined doctor/pharmacy/hospital counts

In [207]:
# Create stratification variables by combining RUCA, REPL, SVI, Healthcare & Resource Access, and Cluster variables into meaningful categories for stratified analyses in R. This will allow us to analyze WER and coherence across different geographic and social vulnerability contexts.
import re


if 'X_location_final' in locals():
    print("="*80)
    print("CREATING STRATIFICATION VARIABLES")
    print("="*80)
    
    X_stratified = X_location_final.copy()
    
    # 1. URBAN/RURAL STRATIFICATION FROM RUCA CODES
    # RUCA codes: 1-3 = Urban, 4-6 = Suburban, 7-10 = Rural
    # ToDo: Erin got these RUCA bins from a Google suggestion on what's been used before
    # https://doh.wa.gov/sites/default/files/legacy/Documents/1500/RUCAGuide.pdf

    print("\n1. Urban/Rural Classification from RUCA Codes")
    print("-" * 80)
    
    if 'PrimaryRUCA' in merged_df.columns:

        
        def classify_urban_rural(ruca_code):
            """Classify RUCA code into urban/suburban/rural categories."""
            if pd.isna(ruca_code):
                return 'Unknown'
            s = str(ruca_code).strip()
            # grab first numeric portion (integer or float, optionally negative)
            m = re.search(r'(-?\d+(?:\.\d+)?)', s)
            if not m:
                return 'Unknown'
            try:
                val = float(m.group(1))
            except ValueError:
                return 'Unknown'
        
            if val <= 1:
                return 'UrbanCore'
            elif val <= 3:
                return 'Suburban'
            elif val <= 6:
                return 'LargeRural'
            elif val <= 10:
                return 'SmallTownRural'
            else:
                return 'Rural'

        X_stratified['urban_rural_category'] = merged_df['PrimaryRUCA'].apply(classify_urban_rural)
        # create the categorical column

        # one‑hot encode it and join back to X_stratified
        dummies = pd.get_dummies(X_stratified['urban_rural_category'],
                                 prefix='urban_rural',
                                 dummy_na=False)   # drop NaN column if you don’t want one
        
        # either overwrite X_stratified or create a new dataframe
        X_stratified = pd.concat([X_stratified, dummies], axis=1)
        print(f"Created urban/rural categories and one-hot encoded them. New shape: {X_stratified.shape}")
        print(X_stratified.head())

        # Also create binary urban vs non-urban
        X_stratified['is_urban'] = (merged_df['PrimaryRUCA'] <= 3).astype(float)
        
        print("Distribution of Urban/Rural Categories:")
        print(X_stratified['urban_rural_category'].value_counts())
        print(f"\nUrban participants: {X_stratified['is_urban'].sum():.0f} ({X_stratified['is_urban'].mean()*100:.1f}%)")
    else:
        print("WARNING: PrimaryRUCA not found in merged_df, skipping urban/rural classification")

    
    # 2. SVI VULNERABILITY STRATIFICATION
    # RPL_THEMES: 0-0.33 = Low vulnerability, 0.33-0.66 = Medium, 0.66-1.0 = High
    # ToDo: Base theses off SVI research literature, want to make sure informed categorizations
    # https://hqin.org/wp-content/uploads/2022/09/Social-Vulnerability-Index-Toolkit.pdf
    print("\n2. SVI Vulnerability Levels")
    print("-" * 80)
    
    if 'RPL_THEMES' in X_location_final.columns:
        def classify_svi_level(svi_score):
            """Classify SVI score into low/medium/high vulnerability"""
            if pd.isna(svi_score):
                return 'Unknown'
            elif svi_score < 0.33:
                return 'Low Vulnerability'
            elif svi_score < 0.66:
                return 'Medium Vulnerability'
            else:
                return 'High Vulnerability'
        
        X_stratified['svi_category'] = X_location_final['RPL_THEMES'].apply(classify_svi_level)
        
        # Also create binary high vulnerability indicator
        X_stratified['high_svi'] = (X_location_final['RPL_THEMES'] >= 0.66).astype(float)
        
        print("Distribution of SVI Categories:")
        print(X_stratified['svi_category'].value_counts())
        print(f"\nHigh vulnerability areas: {X_stratified['high_svi'].sum():.0f} ({X_stratified['high_svi'].mean()*100:.1f}%)")
        
        # Also create SVI theme-specific categorizations
        svi_themes = ['RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4']
        available_themes = [theme for theme in svi_themes if theme in X_location_final.columns]
        
        for theme in available_themes:
            theme_name = theme.replace('RPL_THEME', 'svi_theme')
            X_stratified[f'{theme_name}_high'] = (X_location_final[theme] >= 0.66).astype(float)
    else:
        print("WARNING: RPL_THEMES not found, skipping SVI classification")
    
    # 3. HEALTHCARE ACCESS STRATIFICATION
    # Combine doctor, pharmacy, and hospital counts
    # ToDo: Don't combine these for now. Run analyses separately with just counts. There are issues with combining counts of different types of POIs
    # Could also turn into PCA - Principal Component Analysis - to create composite access score. Summarize multiple access indicators
    print("\n3. Healthcare Access Levels")
    print("-" * 80)
    
    healthcare_cols = ['doctor_count', 'pharmacy_count', 'hospital_count']
    available_healthcare = [col for col in healthcare_cols if col in X_location_final.columns]
    
    if available_healthcare:
        # Create composite healthcare access score
        X_stratified['healthcare_access_score'] = X_location_final[available_healthcare].sum(axis=1)
        
        # Categorize into tertiles: Low/Medium/High access
        healthcare_tertiles = X_stratified['healthcare_access_score'].quantile([0.33, 0.66])
        
        def classify_healthcare_access(score):
            """Classify healthcare access score into tertiles"""
            if pd.isna(score):
                return 'Unknown'
            elif score <= healthcare_tertiles.iloc[0]:
                return 'Low Healthcare Access'
            elif score <= healthcare_tertiles.iloc[1]:
                return 'Medium Healthcare Access'
            else:
                return 'High Healthcare Access'
        
        X_stratified['healthcare_access_category'] = X_stratified['healthcare_access_score'].apply(classify_healthcare_access)
        
        # Also create binary low access indicator
        X_stratified['low_healthcare_access'] = (X_stratified['healthcare_access_score'] <= healthcare_tertiles.iloc[0]).astype(float)
        
        print(f"Healthcare tertile thresholds: Low<={healthcare_tertiles.iloc[0]:.1f}, Medium<={healthcare_tertiles.iloc[1]:.1f}")
        print("\nDistribution of Healthcare Access Categories:")
        print(X_stratified['healthcare_access_category'].value_counts())
        print(f"\nLow healthcare access: {X_stratified['low_healthcare_access'].sum():.0f} ({X_stratified['low_healthcare_access'].mean()*100:.1f}%)")
    else:
        print("WARNING: No healthcare count variables found, skipping healthcare access classification")
    
    # 4. RESOURCE ACCESS STRATIFICATION
    # Combine supermarket, park, bus station counts
    # ToDo: just do these with counts? Don't combine just yet
    print("\n4. Community Resource Access Levels")
    print("-" * 80)
    
    resource_cols = ['supermarket_count', 'park_count', 'bus_station_count']
    available_resources = [col for col in resource_cols if col in X_location_final.columns]
    
    if available_resources:
        # Create composite resource access score
        X_stratified['resource_access_score'] = X_location_final[available_resources].sum(axis=1)
        
        # Categorize into tertiles
        resource_tertiles = X_stratified['resource_access_score'].quantile([0.33, 0.66])
        
        def classify_resource_access(score):
            """Classify resource access score into tertiles"""
            if pd.isna(score):
                return 'Unknown'
            elif score <= resource_tertiles.iloc[0]:
                return 'Low Resource Access'
            elif score <= resource_tertiles.iloc[1]:
                return 'Medium Resource Access'
            else:
                return 'High Resource Access'
        
        X_stratified['resource_access_category'] = X_stratified['resource_access_score'].apply(classify_resource_access)
        
        # Also create binary low access indicator
        X_stratified['low_resource_access'] = (X_stratified['resource_access_score'] <= resource_tertiles.iloc[0]).astype(float)
        
        print(f"Resource tertile thresholds: Low<={resource_tertiles.iloc[0]:.1f}, Medium<={resource_tertiles.iloc[1]:.1f}")
        print("\nDistribution of Resource Access Categories:")
        print(X_stratified['resource_access_category'].value_counts())
        print(f"\nLow resource access: {X_stratified['low_resource_access'].sum():.0f} ({X_stratified['low_resource_access'].mean()*100:.1f}%)")
    else:
        print("WARNING: No resource count variables found, skipping resource access classification")
    
    # 5. COMBINED STRATIFICATIONS
    # Create useful combinations for interaction analyses
    print("\n5. Combined Stratification Variables")
    print("-" * 80)
    
    if 'urban_rural_category' in X_stratified.columns and 'svi_category' in X_stratified.columns:
        X_stratified['urban_high_svi'] = ((X_stratified['urban_rural_category'] == 'Urban') & 
                                           (X_stratified['svi_category'] == 'High Vulnerability')).astype(float)
        X_stratified['rural_high_svi'] = ((X_stratified['urban_rural_category'] == 'Rural') & 
                                           (X_stratified['svi_category'] == 'High Vulnerability')).astype(float)
        
        print(f"Urban + High SVI: {X_stratified['urban_high_svi'].sum():.0f} participants")
        print(f"Rural + High SVI: {X_stratified['rural_high_svi'].sum():.0f} participants")
    
    if 'healthcare_access_category' in X_stratified.columns and 'svi_category' in X_stratified.columns:
        X_stratified['low_healthcare_high_svi'] = ((X_stratified['healthcare_access_category'] == 'Low Healthcare Access') & 
                                                    (X_stratified['svi_category'] == 'High Vulnerability')).astype(float)
        
        print(f"Low Healthcare + High SVI: {X_stratified['low_healthcare_high_svi'].sum():.0f} participants")


    # Remove columns
    X_stratified = X_stratified.drop(columns=['svi_category', 'park_count', 'high_svi', 
                                              'bus_station_count', 'supermarket_count', 'doctor_count', 'pharmacy_count', 'park_count',
                                              'cluster', 
                                              'is_urban', 
                                              # 'urban_rural_category'
                                              'sds-1-work-school-disruption', 'sds-2-social-leisure-disruption', 'sds-3-home-family-disruption', 
                                              'sds-4-school-work-missed', 'sds-5-less-productive-days', 'sds-6-2t-worked-for-other-reasons',
                                              #ToDo: question about why dropping SNR?
                                              'snr',
                                              'PrimaryRUCA_10.0', 'PrimaryRUCA_1.0', 'PrimaryRUCA_2.0', 'PrimaryRUCA_3.0', 'PrimaryRUCA_4.0',
                                              'PrimaryRUCA_5.0', 'PrimaryRUCA_6.0', 'PrimaryRUCA_7.0', 'PrimaryRUCA_8.0',
                                              'PrimaryRUCA_9.0', 'PrimaryRUCA_nan',
                                              'urban_high_svi', 'rural_high_svi', 'low_healthcare_high_svi',
                                              'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4', 'RPL_THEMES'], errors='ignore')



# In your Python notebook, before saving X_stratified:
columns_to_drop = [
    # Drop urban_rural boolean dummies (keep urban_rural_category)
    'urban_rural_LargeRural',
    'urban_rural_SmallTownRural', 
    'urban_rural_Suburban',
    'urban_rural_Unknown',
    'urban_rural_UrbanCore',
    
    # Optionally drop redundant healthcare/resource access boolean indicators
    # (keep the _category and _score versions)
    'low_healthcare_access',
    'low_resource_access',
    # Drop the sheehan disability scale subscores, have sds_total for overall disability level, and the individual subscores are highly correlated with each other and with total, so not adding much value and causing multicollinearity issues in regression analyses
    'sds-1-work-school-disruption', 
    'sds-2-social-leisure-disruption', 
    'sds-3-home-family-disruption', 
    'sds-4-school-work-missed', 
    'sds-5-less-productive-days', 
    'sds-6-2t-worked-for-other-reasons', 

    #Drop the columns that came back as high VIF, which was the self reported substance use
    'opioids-opiates_0.0',  # VIF = 25.22 (binary flag: 0/1)
    'substance_marijuana_0.0',  # VIF = 12.58
    'substance_alcohol_0.0',  # VIF = 12.55
    'substance_stimulant_0.0',  # VIF = 15.22
    'substance_nicotine_0.0',  # VIF = 6.42
    'substance_psychoactive_0.0',  # VIF = 144.93
    'opioids-opiates',  
    'substance_marijuana',  
    'substance_alcohol',
    'substance_stimulant',
    'substance_nicotine',
    'substance_psychoactive',
    
    'PHQ9_high',
    # Keeping phq9-9-suicidal-thoughts' in per Justin Tauscher 4/24/2026, as a clinical signal
    #'phq9-9-suicidal-thoughts',
    'suicidal-thoughts',
    'phq9-suicidal-thoughts',
    'SCL9_moderate', # VIF = 34
    'SCL9_High',
    'SDS_High', # VIF = 3.229971, only moderate but dropping because keeping SDS_total, and the subscore is highly correlated with total and other subscores, 
    'hpsvq-total-score', # When all 3 together VIF Scores=  hpsvq.total.score: 229.284902, hpsvq.voice.score: 101.566976, hpsvq.distress.score: 119.726253
    "hpsvq-total-score",
    # One Hot Encoding Removes, categorical variables,
    'urban_rural_category',
    'healthcare_access_category',
    'resource_access_category',
    'gender_5.0',
    'urban_rural_categoryUnknown',
    # Changye: drop SNR Signal to Noise Ratio, collinearity with WER
    'snr',
    #ToDo: ask Feng about segment count, it's a bit of a weird variable that we created for the location dataset, not sure if it should be included in the stratified dataset or not. It's also highly correlated with total pause duration and pause duration per segment
    "segment_count", 
    "pause_proportion"

]

X_stratified = X_stratified.drop(columns=columns_to_drop, errors='ignore')
remaining_suicidal_cols = [
    c for c in ['phq9-9-suicidal-thoughts', 'suicidal-thoughts', 'phq9-suicidal-thoughts']
    if c in X_stratified.columns
]
print(f"Remaining suicidal-thoughts columns after drop: {remaining_suicidal_cols}")

# SAVE STRATIFIED DATASET
print("\n" + "="*80)
print("SAVING STRATIFIED DATASET")
print("="*80)

X_stratified.to_csv(out_folder + "X_basic_plus_clin_sdh_location_stratified.csv")
print(f"\nSaved stratified dataset: {X_stratified.shape}")
print(f"New stratification columns added: {len(X_stratified.columns) - len(X_location_final.columns)}")

# Display summary of all stratification variables
stratification_vars = [col for col in X_stratified.columns if col not in X_location_final.columns]
print(f"\nStratification variables created ({len(stratification_vars)}):")
for var in stratification_vars:
    print(f"  - {var}")


CREATING STRATIFICATION VARIABLES

1. Urban/Rural Classification from RUCA Codes
--------------------------------------------------------------------------------
Created urban/rural categories and one-hot encoded them. New shape: (2832, 91)
    pid  race_2.0  race_4.0  race_5.0  race_6.0  race_999.0  gender_1.0  \
0  1870         0         0         0         0           0           0   
1  1870         0         0         0         0           0           0   
2  1553         0         0         0         1           0           0   
3  1788         0         0         0         1           0           0   
4  2319         0         0         0         0           0           1   

   gender_3.0  gender_4.0  gender_5.0  ...  recording_duration  word_count  \
0           0           0           0  ...             172.311       400.0   
1           0           0           0  ...             179.702       359.0   
2           0           0           0  ...              42.863        71.0

In [208]:
# ToDo: remove, data check cell. Causing aliasing problems when running Sandwich_script
print("urban_rural_categoryUnknown" in X_stratified.columns)

print(X_stratified.columns.tolist())

False
['pid', 'race_2.0', 'race_4.0', 'race_5.0', 'race_6.0', 'race_999.0', 'gender_1.0', 'gender_3.0', 'gender_4.0', 'binned_age_0.0', 'binned_age_1.0', 'binned_age_3.0', 'binned_age_4.0', 'Y_WER', 'Y_COH', 'AMOS', 'phq9-total', 'scl-avg-global-score', 'phq9-9-suicidal-thoughts', 'dx_group_smi', 'dx_group_substance', 'dx_group_ptsd', 'dx_group_neuro_med', 'dx_group_none', 'hpsvq-voice-score', 'hpsvq-distress-score', 'SDS_Total', 'sexuality_2.0', 'sexuality_3.0', 'sexuality_4.0', 'sexuality_999.0', 'employment-status_1.0', 'employment-status_2.0', 'employment-status_999.0', 'education_binned_2.0', 'education_binned_3.0', 'all_types_drug_use', 'PrimaryRUCA___MISSING__', 'total_pause_duration', 'pause_duration_per_segment', 'recording_duration', 'word_count', 'mean_pause_duration', 'speech_rate', 'svi_theme1_high', 'svi_theme2_high', 'svi_theme3_high', 'svi_theme4_high', 'healthcare_access_score', 'resource_access_score']
